# Simple ProsumerGrid Example

Run the default environment, inspect its Jumanji outputs, try a short rollout, and validate registry and configuration-driven construction.

In [3]:
import platform
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

Python: 3.11.15
Platform: macOS-26.5.2-arm64-arm-64bit


In [ ]:
import jax
import jumanji

import gll_env

print(f"JAX: {jax.__version__}")
print(f"Jumanji: {jumanji.__version__}")
print(f"gll_env ProsumerGrid: {gll_env.ProsumerGrid}")

JAX: 0.5.3
Jumanji: 1.1.2
gll_env ProsumerGrid: <class 'gll_env.env.ProsumerGrid'>


## Default Environment

Construct the environment with its bundled default scenario.

In [5]:
env = gll_env.ProsumerGrid(time_limit=96)

print(env)
print("Action spec:", env.action_spec)
print("Observation spec:", env.observation_spec)

ProsumerGrid(num_agents=18, time_limit=96)
Action spec: BoundedArray(shape=(18, 2), dtype=dtype('float32'), name='action', minimum=Array(-1., dtype=float32), maximum=Array(1., dtype=float32))
Observation spec: observation(
	agents_view=Array(shape=(18, 22), dtype=dtype('float32'), name='agents_view'),
	action_mask=Array(shape=(18, 2), dtype=dtype('bool'), name='action_mask'),
	global_state=Array(shape=(18, 384), dtype=dtype('float32'), name='global_state'),
	action_constraints=action_constraints(
	halfspace_a=Array(shape=(18, 2, 2), dtype=dtype('float32'), name='halfspace_a'),
	halfspace_b=Array(shape=(18, 2), dtype=dtype('float32'), name='halfspace_b'),
	ball_center=Array(shape=(18, 2, 2), dtype=dtype('float32'), name='ball_center'),
	ball_radius=Array(shape=(18, 2), dtype=dtype('float32'), name='ball_radius'),
),
	step_count=Array(shape=(18,), dtype=dtype('int32'), name='step_count'),
)


## Reset And Inspect

A reset returns the Jumanji timestep together with the full typed environment state.

In [6]:
state, timestep = env.reset(jax.random.PRNGKey(0))

print("State fields:", state)
print("Observation:", timestep.observation)
print("Reward:", timestep.reward)
print("Discount:", timestep.discount)
print("Step type:", timestep.step_type)

State fields: EnvironmentState(time_state=DaytimeState(interval_start=Array(0.14583333, dtype=float32), interval_end=Array(0.15625, dtype=float32), day_step=Array(14, dtype=int32)), grid_state=GridState(bus_voltage_pu=Array([1.        +0.j        , 0.86044043-0.49806565j,
       0.8593888 -0.49505946j, 0.85851586-0.49255764j,
       0.8578987 -0.49028233j, 0.85729194-0.48880088j,
       0.8565203 -0.48752704j, 0.8558649 -0.48723948j,
       0.8553627 -0.48722357j, 0.85509145-0.48670474j,
       0.8548895 -0.48631322j, 0.8583128 -0.49097008j,
       0.85859287-0.4895666j , 0.8583818 -0.49028018j,
       0.8583282 -0.48887128j, 0.8575871 -0.48960754j,
       0.856769  -0.48568353j, 0.8550894 -0.48514003j,
       0.8547335 -0.4848422j ], dtype=complex64), bus_power_injection_pu=Array([ 0.03021502+0.06499481j, -0.00013303+0.00902309j,
       -0.00504666-0.01232867j, -0.00130514+0.00341365j,
       -0.00302819-0.01454777j,  0.00385562-0.00531845j,
       -0.00064215-0.01288267j, -0.00407577

## Short Random-Action Rollout

Sample actions from the environment spec and advance a few intervals.

In [7]:
rollout_key = jax.random.PRNGKey(1)

for step in range(5):
    rollout_key, action_key = jax.random.split(rollout_key)
    action = jax.random.uniform(
        action_key,
        shape=env.action_spec.shape,
        minval=env.action_spec.minimum,
        maxval=env.action_spec.maximum,
    )
    state, timestep = env.step(state, action)
    print(
        f"step={step + 1}, reward_shape={timestep.reward.shape}, terminated={bool(timestep.last())}"
    )
    if bool(timestep.last()):
        break

step=1, reward_shape=(18,), terminated=False
step=2, reward_shape=(18,), terminated=False
step=3, reward_shape=(18,), terminated=False
step=4, reward_shape=(18,), terminated=False
step=5, reward_shape=(18,), terminated=False


## Registry Construction

Importing `gll_env` registers the environment with Jumanji.

In [8]:
env2 = jumanji.make("ProsumerGrid-v0")
state2, timestep2 = env2.reset(jax.random.PRNGKey(2))

print(env2)
print("State type:", type(state2).__name__)
print("Timestep type:", type(timestep2).__name__)
print("Observation type:", type(timestep2.observation).__name__)

ProsumerGrid(num_agents=18, time_limit=None)
State type: EnvironmentState
Timestep type: TimeStep
Observation type: MarlObservation


## Configuration-Driven Construction

Build the same environment through `ConfigGenerator` with a minimal valid OmegaConf configuration.

In [9]:
from omegaconf import OmegaConf

config = OmegaConf.create(
    {
        "grid_model": "cigre_lv_consumer",
        "prosumer": {
            "s_pq_max_kVA": 20.0,
            "load": {
                "daily_consumption_kWh": 15.0,
                "s_load_max_kVA": 15.0,
            },
            "inverter": {
                "s_inv_max_kVA": 15.0,
                "battery": {
                    "capacity_kWh": 10.0,
                    "peak_charge_kW": 5.0,
                    "peak_discharge_kW": 5.0,
                },
                "solar": {"peak_power_kW": 8.0},
            },
        },
    }
)

generator = gll_env.ConfigGenerator(
    n_steps_per_day=96,
    grid=config,
    prosumer=config.prosumer,
)
env3 = gll_env.ProsumerGrid(
    generator=generator,
    observer=gll_env.MarlObserver(generator.env_dynamics),
    time_limit=96,
)
state3, timestep3 = env3.reset(jax.random.PRNGKey(3))
print(env3)
print("Configured observation type:", type(timestep3.observation).__name__)

ProsumerGrid(num_agents=18, time_limit=96)
Configured observation type: MarlObservation
